In [ ]:
import tensorflow as tf
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np
import platform
import datetime
import os
import math
import random

print('Python version:', platform.python_version())
print('Tensorflow version:', tf.__version__)
print('Keras version:', tf.keras.__version__)

%load_ext tensorboard
!rm -rf ./logs/

from google.colab import drive
drive.mount('/content/drive')

drive.mount("/content/drive", force_remount = True)

!mkdir /content/train

!cp -r /content/drive/MyDrive/rock_paper_scissors_dataset /content/train

import os
import shutil

# path to the root folder containing class folders
dataset_dir = '/content/train/rock_paper_scissors_dataset'  # should contain subfolders: rock, paper, scissors

# go through each class folder and rename all files
for label in os.listdir(dataset_dir):
    label_path = os.path.join(dataset_dir, label)
    if not os.path.isdir(label_path):
        continue  # skip if not a folder

    for i, filename in enumerate(sorted(os.listdir(label_path))):
        file_path = os.path.join(label_path, filename)

        # skip non-image files
        if not filename.lower().endswith(('.jpg', '.jpeg', '.png')):
            continue

        # new name format: rock0.jpg, rock1.jpg, ...
        new_name = f"{label}{i}.jpg"
        new_path = os.path.join(dataset_dir, new_name)

        # move image to parent folder with new name
        os.rename(file_path, new_path)

    # remove the now-empty class folder (or non-empty if there were non-image files)
    # Use shutil.rmtree for robustness as os.rmdir only removes empty directories.
    shutil.rmtree(label_path)

    import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from PIL import Image, UnidentifiedImageError
import re

# settings
data_dir = '/content/train/rock_paper_scissors_dataset'
crop_height, crop_width = 300, 300

from PIL import ImageOps

# function to Resize an image to fit within the target size while preserving aspect ratio
def resize_and_pad(image: Image.Image, target_size=(300, 300), background_color=(255, 255, 255)):
    # Resize while keeping aspect ratio
    image.thumbnail(target_size, Image.LANCZOS)
    # Create a white background canvas
    canvas = Image.new("RGB", target_size, background_color)
    # Paste the image at the center
    offset_x = (target_size[0] - image.width) // 2
    offset_y = (target_size[1] - image.height) // 2
    canvas.paste(image, (offset_x, offset_y))
    return np.array(canvas, dtype=np.uint8)


# load and preprocess images and labels
image_list = []
label_list = []

for filename in os.listdir(data_dir):
    if filename.lower().endswith(('.png', '.jpg', '.jpeg')):
        path = os.path.join(data_dir, filename)
        try:
            image = Image.open(path).convert('RGB')
        except (OSError, UnidentifiedImageError) as e:
            print(f"Skipping corrupted image: {path} - {e}")
            continue

        image_np = np.array(image)

        if image_np.shape[0] < crop_height or image_np.shape[1] < crop_width:
            continue

        # Reopen image for resize_and_pad to avoid issues with already loaded/closed images
        try:
            image_pil = Image.open(path).convert('RGB')
        except (OSError, UnidentifiedImageError) as e:
            print(f"Skipping corrupted image during resize: {path} - {e}")
            continue

        image_array = resize_and_pad(image_pil, target_size=(300, 300))
        image_list.append(image_array)

        # extract label by removing trailing digits from filename
        raw_label = os.path.splitext(filename)[0]
        label = re.sub(r'\d+$', '', raw_label)
        label_list.append(label)

# encode string labels to integers
unique_labels = sorted(set(label_list))
label_to_index = {label: idx for idx, label in enumerate(unique_labels)}
index_to_label = {v: k for k, v in label_to_index.items()}

def get_label_name(label_id):
    return index_to_label[label_id]

label_indices = np.array([label_to_index[label] for label in label_list], dtype=np.int64)

# convert image list to numpy array
images_array = np.array(image_list, dtype=np.uint8)

# split into training and validation sets
X_train, X_validation, y_train, y_validation = train_test_split(
    images_array, label_indices, test_size=0.2, random_state=42
)

# convert to tf.data.Dataset and shuffle
dataset_train_raw = (
    tf.data.Dataset.from_tensor_slices((X_train, y_train))
    .shuffle(buffer_size=len(X_train), seed=42)
    .prefetch(tf.data.AUTOTUNE)
)

dataset_validation_raw = (
    tf.data.Dataset.from_tensor_slices((X_validation, y_validation))
    .shuffle(buffer_size=len(X_validation), seed=42)
    .prefetch(tf.data.AUTOTUNE)
)

# print dataset structure and size
print('raw training dataset:', dataset_train_raw)
print('training set size:', len(list(dataset_train_raw)), '\n')

print('raw validation dataset:', dataset_validation_raw)
print('validation set size:', len(list(dataset_validation_raw)), '\n')

import collections

# define a simple dataset_info object using namedtuple
DatasetInfo = collections.namedtuple("DatasetInfo", [
    "features", "splits", "num_classes", "label_names", "description"
])

dataset_info = DatasetInfo(
    features={
        "image": tf.TensorSpec(shape=(300, 300, 3), dtype=tf.uint8),
        "label": tf.TensorSpec(shape=(), dtype=tf.int64)
    },
    splits={
        "train": len(X_train),
        "validation": len(X_validation)
    },
    num_classes=len(unique_labels),
    label_names=unique_labels,
    description=f"custom dataset loaded from /content/data with {len(unique_labels)} classes."
)

# extract metadata from dataset_info
NUM_TRAIN_EXAMPLES = dataset_info.splits["train"]
NUM_VALIDATION_EXAMPLES = dataset_info.splits["validation"]
NUM_CLASSES = dataset_info.num_classes

# print summary
print('number of training examples:', NUM_TRAIN_EXAMPLES)
print('number of validation examples:', NUM_VALIDATION_EXAMPLES)
print('number of label classes:', NUM_CLASSES)

dataset_info.features

# extract input image shape and size from dataset_info
INPUT_IMG_SHAPE_ORIGINAL = dataset_info.features['image'].shape
INPUT_IMG_SIZE_ORIGINAL = INPUT_IMG_SHAPE_ORIGINAL[0]  # assumes square image

# define reduced shape by halving original size
INPUT_IMG_SIZE_REDUCED = INPUT_IMG_SIZE_ORIGINAL // 2
INPUT_IMG_SHAPE_REDUCED = (
    INPUT_IMG_SIZE_REDUCED,
    INPUT_IMG_SIZE_REDUCED,
    INPUT_IMG_SHAPE_ORIGINAL[2]  # channel count
)

# set active input shape to reduced version
INPUT_IMG_SIZE = INPUT_IMG_SIZE_REDUCED
INPUT_IMG_SHAPE = INPUT_IMG_SHAPE_REDUCED

# print shape and size information
print('input image size (original):', INPUT_IMG_SIZE_ORIGINAL)
print('input image shape (original):', INPUT_IMG_SHAPE_ORIGINAL)
print()
print('input image size (reduced):', INPUT_IMG_SIZE_REDUCED)
print('input image shape (reduced):', INPUT_IMG_SHAPE_REDUCED)
print()
print('input image size:', INPUT_IMG_SIZE)
print('input image shape:', INPUT_IMG_SHAPE)

print(get_label_name(0));
print(get_label_name(1));
print(get_label_name(2));

for i in range(0,50):
  print(label_list[i])

  def preview_dataset(dataset):
    plt.figure(figsize=(12, 12))
    plot_index = 0
    for features in dataset.take(100):
        (image, label) = features
        plot_index += 1
        plt.subplot(10, 10, plot_index)
        # plt.axis('Off')
        label = get_label_name(label.numpy())
        plt.title('Label: %s' % label)
        plt.imshow(image.numpy())

preview_dataset(dataset_train_raw)

(first_image, first_lable) = list(dataset_train_raw.take(1))[0]
print('Label:', first_lable.numpy(), '\n')
print('Image shape:', first_image.numpy().shape, '\n')
print(first_image.numpy())

def format_example(image, label):
    # convert image to float32
    image = tf.cast(image, tf.float32)
    # normalize pixel values to [0, 1]
    image = image / 255.0
    # resize to the input size expected by the model
    image = tf.image.resize(image, [INPUT_IMG_SIZE, INPUT_IMG_SIZE])
    return image, label

dataset_train = dataset_train_raw.map(format_example)
dataset_validation = dataset_validation_raw.map(format_example)

model1 = tf.keras.models.Sequential()

# First convolution.
model1.add(tf.keras.layers.Convolution2D(
    input_shape=INPUT_IMG_SHAPE,
    filters=64,
    kernel_size=3,
    activation=tf.keras.activations.relu
))
model1.add(tf.keras.layers.MaxPooling2D(
    pool_size=(2, 2),
    strides=(2, 2)
))

# Second convolution.
model1.add(tf.keras.layers.Convolution2D(
    filters=64,
    kernel_size=3,
    activation=tf.keras.activations.relu
))
model1.add(tf.keras.layers.MaxPooling2D(
    pool_size=(2, 2),
    strides=(2, 2)
))

# Third convolution.
model1.add(tf.keras.layers.Convolution2D(
    filters=128,
    kernel_size=3,
    activation=tf.keras.activations.relu
))
model1.add(tf.keras.layers.MaxPooling2D(
    pool_size=(2, 2),
    strides=(2, 2)
))

# Fourth convolution.
model1.add(tf.keras.layers.Convolution2D(
    filters=128,
    kernel_size=3,
    activation=tf.keras.activations.relu
))
model1.add(tf.keras.layers.MaxPooling2D(
    pool_size=(2, 2),
    strides=(2, 2)
))

# Flatten the results to feed into dense layers.
model1.add(tf.keras.layers.Flatten())
model1.add(tf.keras.layers.Dropout(0.5))

# 512 neuron dense layer.
model1.add(tf.keras.layers.Dense(
    units=512,
    activation=tf.keras.activations.relu
))

# Output layer.
model1.add(tf.keras.layers.Dense(
    units=len(unique_labels),
    activation=tf.keras.activations.softmax
))

model1.summary()

tf.keras.utils.plot_model(
    model1,
    show_shapes=True,
    show_layer_names=True,
)

model2 = tf.keras.models.Sequential()


## TODO: now write a code for the simple neural network model similar to what we did yesterday!

# Flatten the raw image directly into a vector (no convolutions).
model2.add(tf.keras.layers.Flatten(input_shape=INPUT_IMG_SHAPE))

# First dense (hidden) layer.
model2.add(tf.keras.layers.Dense(units=150,activation=tf.keras.activations.relu,kernel_regularizer=tf.keras.regularizers.l2(0.010)))

# Second dense (hidden) layer.
model2.add(tf.keras.layers.Dense(units=150,activation=tf.keras.activations.relu,kernel_regularizer=tf.keras.regularizers.l2(0.010)))

# Third dense (hidden) layer.
model2.add(tf.keras.layers.Dense(units=150,activation=tf.keras.activations.relu,kernel_regularizer=tf.keras.regularizers.l2(0.010)))

# Output layer.
model2.add(tf.keras.layers.Dense(units=10,activation=tf.keras.activations.softmax))

model2.summary()

tf.keras.utils.plot_model(
   model2,
   show_shapes=True,
   show_layer_names=True,
)

adam_optimizer = tf.keras.optimizers.Adam(learning_rate=0.001)
rmsprop_optimizer = tf.keras.optimizers.RMSprop(learning_rate=0.001)

model1.compile(
    optimizer=rmsprop_optimizer,
    loss=tf.keras.losses.sparse_categorical_crossentropy,
    metrics=['accuracy']
)

adam_optimizer2 = tf.keras.optimizers.Adam(learning_rate=0.001)
rmsprop_optimizer2 = tf.keras.optimizers.RMSprop(learning_rate=0.001)
model2.compile(
    optimizer=rmsprop_optimizer2,
    loss=tf.keras.losses.sparse_categorical_crossentropy,
    metrics=['accuracy']
)

BATCH_SIZE = 32
dataset_train_shuffled = dataset_train.shuffle(buffer_size=1000).batch(BATCH_SIZE)
dataset_validation_shuffled = dataset_validation.batch(BATCH_SIZE)

print(dataset_train_shuffled)
print(dataset_validation_shuffled)

!rm -rf tmp/checkpoints
!rm -rf logs

training_history1 = model1.fit(
    x=dataset_train_shuffled,
    validation_data=dataset_validation_shuffled,
    epochs=15
)

training_history2 = model2.fit(
    x=dataset_train_shuffled,
    validation_data=dataset_validation_shuffled,
    epochs=15
)

# accuracy per category
import numpy as np
from collections import defaultdict

# initialize counters
correct_per_class = defaultdict(int)
total_per_class = defaultdict(int)

# loop through validation dataset
for images, labels in dataset_validation_shuffled:
    predictions = model1.predict(images)
    predicted_labels = np.argmax(predictions, axis=1)
    true_labels = labels.numpy()

    for true, pred in zip(true_labels, predicted_labels):
        total_per_class[true] += 1
        if true == pred:
            correct_per_class[true] += 1

# print accuracy per class
for class_idx in range(len(unique_labels)):
    correct = correct_per_class[class_idx]
    total = total_per_class[class_idx]
    acc = correct / total if total > 0 else 0
    print(f"Accuracy for class '{unique_labels[class_idx]}': {acc:.2f}")

    # initialize counters
correct_per_class = defaultdict(int)
total_per_class = defaultdict(int)

# loop through validation dataset
for images, labels in dataset_validation_shuffled:
    predictions = model2.predict(images)
    predicted_labels = np.argmax(predictions, axis=1)
    true_labels = labels.numpy()

    for true, pred in zip(true_labels, predicted_labels):
        total_per_class[true] += 1
        if true == pred:
            correct_per_class[true] += 1

# print accuracy per class
for class_idx in range(len(unique_labels)):
    correct = correct_per_class[class_idx]
    total = total_per_class[class_idx]
    acc = correct / total if total > 0 else 0
    print(f"Accuracy for class '{unique_labels[class_idx]}': {acc:.2f}")

import matplotlib.pyplot as plt
history_dict = training_history1.history
accuracy_values = history_dict["accuracy"]
val_accuracy_values = history_dict["val_accuracy"]
epochs = range(1, len(accuracy_values) + 1)
plt.plot(epochs, accuracy_values, "bo", label="Training accuracy")
plt.plot(epochs, val_accuracy_values, "b", label="Validation accuracy")

history_dict2 = training_history2.history
accuracy_values = history_dict2["accuracy"]
val_accuracy_values = history_dict2["val_accuracy"]
epochs = range(1, len(accuracy_values) + 1)
plt.plot(epochs, accuracy_values, "bo", label="Training accuracy 2")
plt.plot(epochs, val_accuracy_values, "b", label="Validation accuracy 2")

plt.title("Training and validation accuracy")
plt.xlabel("Epochs")
plt.ylabel("accuracy")
plt.legend()
plt.show()

# %%capture
train_loss1, train_accuracy1 = model1.evaluate(
    x=dataset_train.batch(BATCH_SIZE).take(NUM_TRAIN_EXAMPLES)
)

validation_loss1, validation_accuracy1 = model1.evaluate(
    x=dataset_validation.batch(BATCH_SIZE).take(NUM_VALIDATION_EXAMPLES)
)

print('training loss:', train_loss1)
print('training accuracy:', train_accuracy1)
print()
print('validation loss:', validation_loss1)
print('validation accuracy:', validation_accuracy1)

train_loss2, train_accuracy2 = model2.evaluate(
    x=dataset_train.batch(BATCH_SIZE).take(NUM_TRAIN_EXAMPLES)
)

validation_loss2, validation_accuracy2 = model2.evaluate(
)

%cd '/content'
model_name1 = 'DEEP_rock_paper_scissors_cnn.keras'
model_name2 = 'DEEP_rock_paper_scissors_ann.keras'

## TODO: try out different hyperparameters and find the best one with the highest validation accuracy
## TODO: pick your best overall model to save here (check all different parameters for model1 and model2)
model1.save(model_name1)
model2.save(model_name2)

model1.save(model_name1, save_format="h5")
model2.save(model_name2, save_format="h5")


